In [1]:
#!pip install gym[Box_2D] Box2D box2d-py gym[box2d]

In [31]:
import gym
#from gym.wrappers import Monitor
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import base64, io

import numpy as np
from collections import deque, namedtuple

# For visualization
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display 
import glob

In [1]:
import gym

### Toy Env

In [87]:
class Two(gym.Env):
    def __init__(self):
        self.action_space = gym.spaces.Discrete(3,start=-1) # Acition = {-1,0,1}  
        self.observation_space = gym.spaces.Discrete(5,start=-2) # State = {-2,-1,0,1,2} 
        self.state = 0 
        self.frame = 0
    def step(self,action):
        self.state = self.state + action
        self.frame = self.frame + 1 
        if self.state == 2:
            reward = 100
        else:
            reward = -1
        info = {}
        if self.state == -2 or self.state==2: 
            done = True
        else: 
            done = False
        return self.state, reward, done, info
    def render(self):
        print('state: {}'.format(self.state))
    def reset(self):
        self.state = 0 
        return self.state

In [88]:
env=Two()

`-` 기능이 있어요 

In [89]:
env.action_space.sample()

1

In [90]:
env.observation_space.sample()

-1

In [91]:
episodes = 3
for episode in range(1, episodes+1):
    state = env.reset()
    done = False 
    score = 0 
    
    while not done: 
        action = env.action_space.sample()
        print("action: {}".format(action))
        n_state, reward, done, info = env.step(action) 
        env.render()
        score = score + reward 
    print("==Episode:{} Score:{}==".format(episode,score))

action: 1
state: 1
action: 1
state: 2
==Episode:1 Score:99==
action: -1
state: -1
action: 0
state: -1
action: 0
state: -1
action: -1
state: -2
==Episode:2 Score:-4==
action: 1
state: 1
action: 0
state: 1
action: -1
state: 0
action: 1
state: 1
action: -1
state: 0
action: 0
state: 0
action: 0
state: 0
action: -1
state: -1
action: -1
state: -2
==Episode:3 Score:-9==


### 약간의 센스를 발휘하면 

In [92]:
state_graphic = {-2:'X....', -1:'.X...', 0:'..X..',1:'...X.', 2:'....X'}

In [93]:
state_graphic[-2]

'X....'

In [94]:
state_graphic[1]

'...X.'

In [95]:
class Two(gym.Env):
    def __init__(self):
        self.action_space = gym.spaces.Discrete(3,start=-1) # Acition = {-1,0,1}  
        self.observation_space = gym.spaces.Discrete(5,start=-2) # State = {-2,-1,0,1,2} 
        self.state = 0 
        self.frame = 0
    def step(self,action):
        self.state = self.state + action
        self.frame = self.frame + 1 
        if self.state == 2:
            reward = 100
        else:
            reward = -1
        info = {}
        if self.state == -2 or self.state==2: 
            done = True
        else: 
            done = False
        return self.state, reward, done, info
    def render(self):
        print(state_graphic[self.state])
    def reset(self):
        self.state = 0 
        return self.state

In [96]:
episodes = 3 
env = Two()
for episode in range(1, episodes+1):
    state = env.reset()
    done = False 
    score = 0 
    
    while not done: 
        action = env.action_space.sample()
        #print("action: {}".format(action))
        n_state, reward, done, info = env.step(action) 
        env.render()
        score = score + reward 
    print("==Episode:{} Score:{}==".format(episode,score))

...X.
....X
==Episode:1 Score:99==
..X..
.X...
.X...
X....
==Episode:2 Score:-4==
..X..
..X..
...X.
..X..
.X...
..X..
.X...
X....
==Episode:3 Score:-8==


### 게임을 클리어 하는 방법?

`-` 너무 쉬움.. 무조건 action = 1 이면 된다. 

In [97]:
episodes = 3 
env = Two()
for episode in range(1, episodes+1):
    state = env.reset()
    done = False 
    score = 0 
    
    while not done: 
        action = 1
        #print("action: {}".format(action))
        n_state, reward, done, info = env.step(action) 
        env.render()
        score = score + reward 
    print("==Episode:{} Score:{}==".format(episode,score))

...X.
....X
==Episode:1 Score:99==
...X.
....X
==Episode:2 Score:99==
...X.
....X
==Episode:3 Score:99==


`-` action = 1 은 어떠한 상태라도 최적의 액션이 된다. 

### 장애물이 있다면?

In [101]:
class Two(gym.Env):
    def __init__(self):
        self.action_space = gym.spaces.Discrete(5,start=-2) # Acition = {-2,-1,0,1,2}  
        self.observation_space = gym.spaces.Discrete(11,start=-5) # State = {-5,-4,-3,-2,-1,0,1,2,3,4,5} 
        self.state = 0 
        self.frame = 0
    def step(self,action):
        self.state = self.state + action
        self.frame = self.frame + 1 
        if self.state == 2:
            reward = 100
        else:
            reward = -1
        info = {}
        if self.state == -2 or self.state==2: 
            done = True
        else: 
            done = False
        return self.state, reward, done, info
    def render(self):
        print(state_graphic[self.state])
    def reset(self):
        self.state = 0 
        return self.state

### 데이터로 정책을 학습한다면 어떨까? 

### Q-learning

In [2]:
class ShowerEnv(gym.Env):
    def __init__(self):
        self.action_space = spaces.Discrete(3) 
        self.observation_space = spaces.Box(low=np.array([0]),high=np.array([100]))
        self.state = 38 + np.random.randint(-3,3)
        self.shower_length = 60
    def step(self,action):
        self.state += action - 1
        self.shower_length -= 1 
        if self.state >=37 and self.state <=39:
            reward = 1 
        else:
            reward = -1 
        if self.shower_length <= 0:
            done = True 
        else: 
            done = False
        self.state += np.random.randint(-1,1) 
        info = {}
        return self.state, reward, done, info 
    def render(self):
        print(self.state)
    def reset(self):
        self.state = 38 + np.random.randint(-3,3)
        self.shower_lenght = 60 
        return self.state 

In [41]:
env = ShowerEnv()

In [42]:
env.render()

40


In [45]:
env.step(1)

(39, -1, False, {})

In [46]:
env.render()

39


In [32]:
env.action_space.sample()

0